# PDF to Audiobook — Chatterbox TTS Voice Cloning

Converts a book into a voice-cloned audiobook, **automatically detecting and
repairing misread or dropped words** until each chapter hits a target accuracy.

## One-click flow (recommended)

Run these once at the start, in order:

| Cell | Purpose |
|------|---------|
| 0 | Pull latest code from GitHub |
| 1 | Check GPU |
| 2 | Install packages → **restart session → re-run Cell 2** |
| 3 | Mount Google Drive |
| 4 | **CONFIG — set all paths here** |
| 4A | Prepare a new book (clean + split a Gutenberg .txt) |
| 5 | Helper functions |
| 6 | Trim reference audio |
| 7 | Load Chatterbox TTS model |
| **P** | **ONE-CLICK PIPELINE — generate + auto-repair the whole book** |
| S | Status (run anytime, even in a 2nd tab) |

**Cell P does everything**: chunks every chapter, generates audio, transcribes
it, compares against the source, and re-rolls any chunk with dropped/misread
content until it reaches the target accuracy — then stitches the full book and
writes a QA report. **It resumes automatically after a Colab disconnect** — just
re-run Cells 0,2,3,4,5,6,7 then P again.

Cells 8–13 below are the older manual/debug tools (single-chapter convert,
WhisperX inspect, manual diff) — optional, not needed for the one-click flow.

**Test books (Project Gutenberg plain text):**
- [The Adventures of Sherlock Holmes](https://www.gutenberg.org/ebooks/1661)
- [The Time Machine](https://www.gutenberg.org/ebooks/35)

In [ ]:
# ── Cell 0: Pull latest code from GitHub ─────────────────────────
# Run this once at the start of every session.
# Opens the repo so all scripts are available in this runtime.

import os
from pathlib import Path

REPO_URL = "https://github.com/Mashfique/Audio_Booker.git"
REPO_DIR = "/content/Audio_Booker"
BRANCH   = "dev"

if Path(REPO_DIR).exists():
    print("Repo already cloned — pulling latest changes...")
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull
else:
    print("Cloning repo...")
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

# Verify clone succeeded before changing directory
if not Path(REPO_DIR).exists():
    raise RuntimeError(
        f"Clone failed — repo not found at {REPO_DIR}\n"
        "If your repo is private, go to GitHub → Settings → Change visibility → Public"
    )

%cd {REPO_DIR}
print(f"\nRepo ready at: {REPO_DIR}  (branch: {BRANCH})")
print("Files available:")
!ls

In [ ]:
# ── Cell 1: Check GPU + Python version ───────────────────────────
import sys, torch

print(f"Python : {sys.version}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU    : {gpu.name}")
    print(f"VRAM   : {gpu.total_memory / 1e9:.1f} GB")
else:
    print("No GPU — go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# ── Cell 2: Install packages ──────────────────────────────────────
%pip install -q chatterbox-tts pdfplumber

# faster-whisper = the ASR used by the one-click pipeline (Cell P).
# CTranslate2-based, so it does NOT trigger the torch circular-import
# that whisperx hits when loaded alongside chatterbox.
%pip install -q faster-whisper

# Fix torchvision to match torch 2.6.0 that chatterbox-tts installs
%pip install -q torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124 --force-reinstall

# Numba (used by librosa, used by chatterbox) needs numpy <= 2.0
# Keep this LAST so numpy ends pinned at 1.26.4
%pip install -q "numpy==1.26.4" --force-reinstall

# !! After this finishes → Runtime → Restart session → re-run this cell → continue from Cell 3

In [ ]:
# ── Cell 3: Mount Google Drive ────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted. Use the folder icon in the left sidebar to browse files.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ── CELL 4: ALL CONFIGURATION — edit this cell before running anything else ──
# ══════════════════════════════════════════════════════════════════════════════
import os
from pathlib import Path

# ── Google Drive base folder ──────────────────────────────────────────────────
# Change this if your audiobook folder has a different name on Drive
DRIVE_BASE = "/content/drive/MyDrive/audiobook"

# ── Input ─────────────────────────────────────────────────────────────────────
TXT_DIR   = f"{DRIVE_BASE}/chapters"       # folder of per-chapter .txt files
VOICE_REF = f"{DRIVE_BASE}/reference.mp3"  # your voice cloning reference clip

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_DIR      = f"{DRIVE_BASE}/output"   # per-chapter MP3s are saved here
FULL_OUTPUT_DIR = f"{DRIVE_BASE}"          # stitched full audiobook saved here

# ── Model cache ───────────────────────────────────────────────────────────────
# Persists the ~1.5 GB model weights on Drive — avoids re-downloading each session
HF_CACHE_DIR = f"{DRIVE_BASE}/hf_cache"

# ── TTS settings ──────────────────────────────────────────────────────────────
REF_SECONDS = 15    # seconds to use from reference audio (6–30 ideal)
CHUNK_WORDS = 200   # words per TTS chunk (keep under 250)

# ── Derived paths — do not edit ───────────────────────────────────────────────
REF_WAV = "/content/reference_trimmed.wav"   # trimmed reference, lives in Colab runtime only

# ── Create all output directories ─────────────────────────────────────────────
for _d in [TXT_DIR, OUTPUT_DIR, FULL_OUTPUT_DIR, HF_CACHE_DIR]:
    Path(_d).mkdir(parents=True, exist_ok=True)

# ── Print summary ─────────────────────────────────────────────────────────────
W = 60
print("═" * W)
print("  CONFIGURATION SUMMARY")
print("═" * W)
print(f"  Drive base       : {DRIVE_BASE}")
print(f"  Chapters (input) : {TXT_DIR}")
print(f"  Voice reference  : {VOICE_REF}")
print(f"  Chapter MP3s out : {OUTPUT_DIR}")
print(f"  Full audiobook   : {FULL_OUTPUT_DIR}")
print(f"  Model cache      : {HF_CACHE_DIR}")
print(f"  Ref WAV (temp)   : {REF_WAV}")
print("─" * W)
print(f"  Reference clip   : {REF_SECONDS}s  |  Chunk size: {CHUNK_WORDS} words")
print("─" * W)
print("  Folders created  : OK (all output directories exist)")
print("═" * W)

In [ ]:
# ── Cell 4A: Prepare a new book (run once per book) ──────────────
# Skip this cell if you already have per-chapter .txt files in TXT_DIR.
#
# 1. Upload your raw .txt file (e.g. from Project Gutenberg) to Drive
# 2. Set RAW_TXT below, then run this cell
# 3. It cleans the text and splits it into numbered chapter files in TXT_DIR

RAW_TXT = f"{DRIVE_BASE}/pg1661.txt"    # ← path to your raw downloaded .txt file

# ── Run tts_cleaner.py ────────────────────────────────────────────
CLEANED_TXT = f"{DRIVE_BASE}/cleaned.txt"
print("Step 1 — Cleaning text...")
!python /content/Audio_Booker/tts_cleaner.py "{RAW_TXT}" "{CLEANED_TXT}"

# ── Run chapter_splitter.py ───────────────────────────────────────
print("\nStep 2 — Splitting into chapters...")
!python /content/Audio_Booker/chapter_splitter.py "{CLEANED_TXT}" "{TXT_DIR}"

print(f"\nDone. Chapter .txt files are in: {TXT_DIR}")
print("You can now run Cells 5 → 6 → 7 → 8 → 9 to start converting.")

In [ ]:
# ── Cell 5: Helper functions ──────────────────────────────────────
import re, subprocess, tempfile, shutil
from pathlib import Path
from collections import Counter
import pdfplumber

CHAPTER_PATTERNS = [
    r"^chapter\s+[\divxlcdm]+", r"^chapter\s+\w+",
    r"^part\s+[\divxlcdm]+",    r"^part\s+\w+",
    r"^\d+\.\s+[A-Z]",
    r"^epilogue$", r"^prologue$", r"^introduction$",
    r"^preface$",  r"^foreword$", r"^appendix", r"^conclusion$",
]

_KNOWN_ACRONYMS = {
    "AI", "ML", "UK", "US", "EU", "UN", "NATO", "FBI", "CIA", "NASA",
    "CEO", "CFO", "CTO", "HR", "IT", "PR", "ID", "OK", "TV", "PC",
    "USB", "PDF", "MP3", "TTS", "GPU", "CPU", "RAM", "API", "URL",
    "HTML", "CSS", "SQL", "GMT", "EST", "PST", "BC", "AD", "WWII", "WWI",
    "USA", "UAE", "PTSD", "DNA", "RNA", "VIP", "RSVP",
}

def _fix_allcaps(text):
    def _replace(m):
        w = m.group(0)
        return w if w in _KNOWN_ACRONYMS else w.capitalize()
    return re.sub(r"\b[A-Z]{3,}\b", _replace, text)

def clean_text(text):
    text = re.sub(r"-\n(\w)", r"\1", text)
    text = re.sub(r"^\s*\d+\s*$", "", text, flags=re.MULTILINE)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", text)
    text = _fix_allcaps(text)
    return text.strip()

def is_chapter_heading(line):
    s = line.strip()
    if not s or len(s) > 80: return False
    return any(re.match(p, s, re.IGNORECASE) for p in CHAPTER_PATTERNS)

def find_repeated_lines(pdf, threshold=5):
    counts = Counter()
    for page in pdf.pages:
        seen = set()
        for line in (page.extract_text() or "").splitlines():
            s = line.strip()
            if s and s not in seen:
                counts[s] += 1; seen.add(s)
    return {l for l, n in counts.items() if n >= threshold}

def extract_chapters(pdf_path):
    chapters, current = [], {"title": "Front Matter", "text": ""}
    with pdfplumber.open(pdf_path) as pdf:
        repeated = find_repeated_lines(pdf)
        for page in pdf.pages:
            for line in (page.extract_text() or "").splitlines():
                s = line.strip()
                if s in repeated: continue
                if is_chapter_heading(s):
                    if current["text"].strip(): chapters.append(current)
                    current = {"title": s, "text": ""}
                else:
                    current["text"] += line + "\n"
    if current["text"].strip(): chapters.append(current)
    return chapters

def split_by_words(text, max_words):
    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks, current, count = [], [], 0
    for s in sentences:
        w = len(s.split())
        if count + w > max_words and current:
            chunks.append(" ".join(current)); current, count = [s], w
        else:
            current.append(s); count += w
    if current: chunks.append(" ".join(current))
    return chunks

def ffmpeg_concat(parts, output):
    with tempfile.TemporaryDirectory() as tmp:
        lst = Path(tmp) / "list.txt"
        lst.write_text("\n".join(f"file '{Path(p).resolve()}'" for p in parts))
        subprocess.run(["ffmpeg", "-y", "-f", "concat", "-safe", "0",
                        "-i", str(lst), "-c", "copy", str(output)],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)

def wav_to_mp3(wav, mp3):
    subprocess.run(["ffmpeg", "-y", "-i", str(wav), "-b:a", "192k", str(mp3)],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    Path(wav).unlink()

print("Helper functions ready.")

In [ ]:
# ── Cell 6: Prepare reference audio ──────────────────────────────
import subprocess

# Trim to REF_SECONDS starting at 3s (skips any intro noise)
subprocess.run(
    ["ffmpeg", "-y", "-i", VOICE_REF,
     "-ss", "3", "-t", str(REF_SECONDS),
     "-ar", "22050", "-ac", "1", REF_WAV],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True
)
print(f"Reference audio trimmed to {REF_SECONDS}s")
print(f"  Source : {VOICE_REF}")
print(f"  Output : {REF_WAV}")

In [ ]:
# ── Cell 7: Load Chatterbox TTS model ────────────────────────────
# Cache model weights to Drive — only downloads once, loads from Drive every session after
import os
os.environ["HF_HOME"] = HF_CACHE_DIR

import torch
from chatterbox.tts import ChatterboxTTS

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading Chatterbox TTS on {device.upper()}...")
print(f"Model cache : {HF_CACHE_DIR}")

model = ChatterboxTTS.from_pretrained(device=device)
print("Model loaded and ready.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ── CELL P: ONE-CLICK AUTONOMOUS PIPELINE ───────────────────────────────────
#   Generates every chapter, transcribes it, compares against the source, and
#   auto-repairs dropped/misread chunks until each hits TARGET_ACCURACY.
#   Resumes automatically after a Colab disconnect — just re-run Cells
#   0,2,3,4,5,6,7 then this cell again. Safe to re-run anytime.
# ══════════════════════════════════════════════════════════════════════════════
#
#   Prerequisites (run once this session): Cell 0, 2, 3, 4, 5, 6, 7
#   and Cell 4A if this is a new book.
#
# ── Tuning (defaults match the agreed design) ─────────────────────────────────
TARGET_ACCURACY = 0.98   # per-chunk word accuracy the loop aims for
BEST_OF         = 3      # candidates generated per attempt, best kept
MAX_RETRIES     = 4      # repair attempts before a chunk is flagged
ASR_MODEL_SIZE  = "small"  # faster-whisper size (small = fewer mishears)

# ─────────────────────────────────────────────────────────────────────────────
import sys, subprocess, torch
from pathlib import Path
sys.path.insert(0, "/content/Audio_Booker")

from audiobook_pipeline import AudiobookPipeline
import scipy.io.wavfile as wavfile
from faster_whisper import WhisperModel

# ── ASR for the loop (CTranslate2 — no torch conflict with Chatterbox) ────────
_use_cuda = torch.cuda.is_available()
try:
    _asr = WhisperModel(
        ASR_MODEL_SIZE,
        device="cuda" if _use_cuda else "cpu",
        compute_type="float16" if _use_cuda else "int8",
    )
except Exception as e:                                    # cuDNN/driver fallback
    print(f"faster-whisper GPU init failed ({e}); falling back to CPU.")
    _asr = WhisperModel(ASR_MODEL_SIZE, device="cpu", compute_type="int8")


def tts_generate(text: str, out_mp3: str) -> None:
    """Chatterbox -> wav -> mp3 (one chunk)."""
    wav_tensor = model.generate(text, audio_prompt_path=REF_WAV)
    audio_np = wav_tensor.squeeze().cpu().numpy()
    tmp_wav = out_mp3 + ".tmp.wav"
    wavfile.write(tmp_wav, model.sr, audio_np)
    subprocess.run(
        ["ffmpeg", "-y", "-i", tmp_wav, "-b:a", "192k", out_mp3],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True,
    )
    Path(tmp_wav).unlink(missing_ok=True)


def asr_transcribe(audio_path: str) -> list:
    """faster-whisper -> flat list of spoken words."""
    segments, _ = _asr.transcribe(audio_path, language="en", beam_size=1)
    words = []
    for seg in segments:
        words.extend(seg.text.split())
    return words


# ── Load chapters ─────────────────────────────────────────────────────────────
txt_files = sorted(Path(TXT_DIR).glob("*.txt"))
if not txt_files:
    raise FileNotFoundError(
        f"No .txt files in {TXT_DIR} — run Cell 4A to prepare the book first."
    )
chapters = [{"title": f.stem, "text": f.read_text(encoding="utf-8")}
            for f in txt_files]
print(f"Loaded {len(chapters)} chapter(s) from {TXT_DIR}")

BOOK_DIR = f"{DRIVE_BASE}/pipeline"
FULL_MP3 = f"{FULL_OUTPUT_DIR}/{Path(TXT_DIR).name}_full.mp3"

pipe = AudiobookPipeline(
    book_dir=BOOK_DIR,
    chapters=chapters,
    tts_generate=tts_generate,
    asr_transcribe=asr_transcribe,
    full_output_path=FULL_MP3,
    target_accuracy=TARGET_ACCURACY,
    best_of=BEST_OF,
    max_retries=MAX_RETRIES,
    chunk_words=CHUNK_WORDS,
)

report = pipe.run()

print("\n" + "═" * 60)
print("  PIPELINE COMPLETE")
print("═" * 60)
print(f"  Overall accuracy   : {report['overall_accuracy']:.1%}")
print(f"  Total chunks       : {report['total_chunks']}")
print(f"  Flagged for review : {report['flagged_count']}")
print(f"  Repair attempts    : {report['repair_attempts_used']}")
print(f"  Full audiobook     : {FULL_MP3}")
print(f"  QA report          : {BOOK_DIR}/qa_report.html")
print("═" * 60)

In [ ]:
# ── CELL S: STATUS — safe to run anytime (even in a 2nd Colab tab) ───────────
# Read-only. Shows where the pipeline is without touching the run.
# Needs Cell 4 (config) to have run so DRIVE_BASE / TXT_DIR exist.

import sys
from pathlib import Path
sys.path.insert(0, "/content/Audio_Booker")
from audiobook_pipeline import AudiobookPipeline

_txt = sorted(Path(TXT_DIR).glob("*.txt"))
_chs = [{"title": f.stem, "text": f.read_text(encoding="utf-8")} for f in _txt]

# Dummy callables — status() never generates or transcribes anything.
_probe = AudiobookPipeline(
    book_dir=f"{DRIVE_BASE}/pipeline",
    chapters=_chs,
    tts_generate=lambda *a, **k: None,
    asr_transcribe=lambda *a, **k: [],
    full_output_path=f"{DRIVE_BASE}/_probe.mp3",
    chunk_words=CHUNK_WORDS,
)
st = _probe.status()

print("═" * 64)
print(f"  Stage     : {st['stage']}")
print(f"  Heartbeat : {st['heartbeat']}")
print(f"  Repairs   : {st['repair_attempts']}")
print("═" * 64)
for c in st["chapters"]:
    title = c["title"][:38]
    if "done" in c:
        bar = f"{c['done']}/{c['chunks']} done"
        extra = (f", {c['flagged']} flagged" if c['flagged'] else "")
        pend = (f", {c['pending']} pending" if c['pending'] else "")
        print(f"  Ch{c['idx']:02d} {title:38s} {bar}{extra}{pend}")
    else:
        print(f"  Ch{c['idx']:02d} {title:38s} {c.get('state','not started')}")
print("═" * 64)

In [ ]:
# ── Cell 8: Load chapters from .txt files ────────────────────────
txt_files = sorted(Path(TXT_DIR).glob("*.txt"))

if not txt_files:
    raise FileNotFoundError(f"No .txt files found in {TXT_DIR}")

chapters = []
for f in txt_files:
    chapters.append({"title": f.stem, "text": f.read_text(encoding="utf-8")})

print(f"Found {len(chapters)} chapter(s):\n")
for i, ch in enumerate(chapters, 1):
    print(f"  {i:02d}. {ch['title']}  ({len(ch['text'].split()):,} words)")

In [ ]:
# ── Cell 9: Convert chapters to MP3 ──────────────────────────────

# ── Chapter selection ─────────────────────────────────────────────
# None  = convert all chapters
# [1,2,3] = convert only chapters 1, 2 and 3 (by number)
CHAPTERS_TO_RUN = None

# ── Notifications ─────────────────────────────────────────────────
# Plays a chime in your browser after each chapter completes.
# Make sure your browser tab is not muted.
NOTIFY = True

# ─────────────────────────────────────────────────────────────────
import numpy as np
import scipy.io.wavfile as wavfile
from tqdm.notebook import tqdm
from IPython.display import Audio, display

def chime(success=True):
    """Play a short tone in the browser to signal chapter completion."""
    if not NOTIFY:
        return
    sr = 22050
    t  = np.linspace(0, 0.4, int(sr * 0.4), endpoint=False)
    if success:
        # Two rising tones — chapter done
        tone = (np.sin(2 * np.pi * 660 * t) * np.exp(-5 * t) +
                np.sin(2 * np.pi * 880 * t) * np.exp(-8 * t)) * 0.3
    else:
        # Low buzz — something went wrong
        tone = np.sin(2 * np.pi * 220 * t) * np.exp(-4 * t) * 0.3
    display(Audio(tone.astype(np.float32), rate=sr, autoplay=True))

out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

# Pre-calculate all chapters and their chunks
all_chapters = []
for i, ch in enumerate(chapters, 1):
    if CHAPTERS_TO_RUN is not None and i not in CHAPTERS_TO_RUN:
        continue
    chunks = split_by_words(ch["text"], CHUNK_WORDS)
    safe   = re.sub(r"[^\w\s-]", "", ch["title"])[:50].strip()
    fname  = f"{i:02d}_{safe}.mp3"
    all_chapters.append((i, ch, chunks, fname))

total_chunks = sum(len(chunks) for _, _, chunks, _ in all_chapters)
done_chunks  = sum(len(chunks) for _, _, chunks, fname in all_chapters
                   if (out_dir / fname).exists())
skipped      = sum(1 for _, _, _, fname in all_chapters if (out_dir / fname).exists())

selected_msg = f"chapters {CHAPTERS_TO_RUN}" if CHAPTERS_TO_RUN else "all chapters"
print(f"Processing {selected_msg} — {len(all_chapters)} chapter(s), {total_chunks} chunks total")
if skipped:
    print(f"Resuming: {skipped} chapter(s) already done, skipping them\n")

overall = tqdm(total=total_chunks, initial=done_chunks,
               desc="Overall", unit="chunk", ncols=70)

for i, ch, chunks, fname in all_chapters:
    out = out_dir / fname

    if out.exists():
        print(f"[{i}/{len(all_chapters)}] SKIP — {ch['title']} (already converted)")
        continue

    overall.set_description(f"Ch {i}/{len(all_chapters)}")
    print(f"\n[{i}/{len(all_chapters)}] {ch['title']} — {len(chunks)} chunk(s)")

    try:
        with tempfile.TemporaryDirectory() as tmp:
            parts = []
            for j, chunk in enumerate(chunks):
                wav = f"{tmp}/chunk_{j:04d}.wav"
                mp3 = f"{tmp}/chunk_{j:04d}.mp3"

                wav_tensor = model.generate(chunk, audio_prompt_path=REF_WAV)
                audio_np = wav_tensor.squeeze().cpu().numpy()
                wavfile.write(wav, model.sr, audio_np)
                wav_to_mp3(wav, mp3)
                parts.append(mp3)
                overall.update(1)

            if len(parts) == 1:
                shutil.copy(parts[0], out)
            else:
                ffmpeg_concat(parts, out)

        size_kb = out.stat().st_size // 1024
        print(f"  → {fname} ({size_kb} KB)")
        chime(success=True)

    except Exception as e:
        print(f"  ✗ ERROR on chapter {i}: {e}")
        chime(success=False)
        raise

overall.close()
print(f"\nDone! {len(all_chapters)} MP3(s) saved to {out_dir}")
chime(success=True)
chime(success=True)  # double chime = fully complete

In [ ]:
# ── Cell 10: (Optional) Stitch all chapters into one file ─────────
from pathlib import Path

book_name = Path(TXT_DIR).name              # e.g. "chapters" or "sherlock_chapters"
full_mp3  = Path(FULL_OUTPUT_DIR) / f"{book_name}_full.mp3"
mp3s      = sorted(Path(OUTPUT_DIR).glob("*.mp3"))

if not mp3s:
    print(f"No MP3s found in {OUTPUT_DIR} — run Cell 9 first.")
else:
    print(f"Stitching {len(mp3s)} chapter(s)...")
    ffmpeg_concat(mp3s, full_mp3)
    size_mb = full_mp3.stat().st_size / (1024 * 1024)
    print(f"Full audiobook : {full_mp3}  ({size_mb:.1f} MB)")

In [ ]:
# ── Cell 12: Transcribe a chapter MP3 with word-level timestamps ──
#
# By default this transcribes the FIRST chapter in OUTPUT_DIR.
# To pick a specific chapter, set CHECK_MP3 manually:
#   CHECK_MP3 = f"{OUTPUT_DIR}/03_A_Scandal_In_Bohemia.mp3"
# Otherwise leave CHECK_MP3 = None and it auto-selects the first file.

CHECK_MP3 = None    # ← set to a path string to pick a specific chapter, or leave None

# ── Auto-detect if not set ────────────────────────────────────────
available = sorted(Path(OUTPUT_DIR).glob("*.mp3"))
if not available:
    raise FileNotFoundError(f"No MP3s found in {OUTPUT_DIR} — run Cell 9 first.")

if CHECK_MP3 is None:
    CHECK_MP3 = str(available[0])
    print(f"Auto-selected: {Path(CHECK_MP3).name}")

# ── List all available chapters ───────────────────────────────────
print("\nAvailable chapter MP3s:")
for f in available:
    marker = "  ◀ checking this" if f == Path(CHECK_MP3) else ""
    print(f"  {f.name}{marker}")
print()

import whisperx, torch, json

device       = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"

print(f"Loading Whisper on {device.upper()}...")
wx_model = whisperx.load_model("base", device, compute_type=compute_type, language="en")

print(f"Transcribing: {Path(CHECK_MP3).name}")
audio  = whisperx.load_audio(CHECK_MP3)
result = wx_model.transcribe(audio, language="en")

print("Aligning word timestamps...")
align_model, metadata = whisperx.load_align_model(language_code="en", device=device)
result = whisperx.align(result["segments"], align_model, metadata, audio, device)

# ── Flatten to word list ──────────────────────────────────────────
words = []
for seg in result["segments"]:
    for w in seg.get("words", []):
        words.append({
            "word":  w.get("word", "").strip(),
            "start": round(w.get("start", 0), 3),
            "end":   round(w.get("end",   0), 3),
            "score": round(w.get("score", 1.0), 3),
        })

# ── Save .transcript.txt ──────────────────────────────────────────
transcript_path = Path(CHECK_MP3).with_suffix(".transcript.txt")
lines = []
for w in words:
    flag = " ⚠" if w["score"] < 0.7 else ""
    lines.append(f"{w['start']:>8.2f}s  {w['word']}{flag}")
transcript_path.write_text("\n".join(lines), encoding="utf-8")
low = sum(1 for w in words if w["score"] < 0.7)
print(f"\nTranscript saved : {transcript_path.name}  ({len(words)} words, {low} flagged)")

# ── Save .html synced player ──────────────────────────────────────
spans = []
for i, w in enumerate(words):
    low_cls = " low" if w["score"] < 0.7 else ""
    text    = w["word"].replace("&", "&amp;").replace("<", "&lt;")
    spans.append(
        f'<span id="w{i}" class="w{low_cls}" '
        f'data-s="{w["start"]}" data-e="{w["end"]}">{text}</span>'
    )
word_data = json.dumps(
    [{"s": w["start"], "e": w["end"]} for w in words],
    separators=(",", ":"),
)

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>{Path(CHECK_MP3).stem}</title>
<style>
  body  {{ font-family: Georgia, serif; max-width: 860px; margin: 40px auto;
           padding: 0 20px; background: #1a1a2e; color: #e0e0e0; }}
  h2    {{ color: #a0c4ff; }}
  audio {{ width: 100%; margin: 16px 0; }}
  #text {{ line-height: 2.2; font-size: 1.15rem; }}
  .w    {{ cursor: pointer; padding: 1px 2px; border-radius: 3px; transition: background 0.1s; }}
  .w:hover  {{ background: #334; }}
  .active   {{ background: #4a90e2; color: #fff; border-radius: 4px; }}
  .low      {{ text-decoration: underline wavy #ff6b6b; }}
  #legend   {{ font-size: 0.85rem; color: #888; margin-top: 12px; }}
</style>
</head>
<body>
<h2>{Path(CHECK_MP3).stem}</h2>
<audio id="audio" controls src="{Path(CHECK_MP3).name}"></audio>
<div id="text">{" ".join(spans)}</div>
<p id="legend">
  <span style="background:#4a90e2;color:#fff;padding:2px 6px;border-radius:3px;">highlighted</span> = currently playing &nbsp;|&nbsp;
  <span style="text-decoration:underline wavy #ff6b6b;">underlined</span> = low confidence (may be mumbled/dropped)
</p>
<script>
const audio = document.getElementById('audio');
const words = {word_data};
let last = -1;
audio.addEventListener('timeupdate', () => {{
  const t = audio.currentTime;
  let idx = -1;
  for (let i = 0; i < words.length; i++) {{
    if (t >= words[i].s && t <= words[i].e) {{ idx = i; break; }}
  }}
  if (idx !== last) {{
    if (last >= 0) document.getElementById('w'+last)?.classList.remove('active');
    if (idx  >= 0) document.getElementById('w'+idx )?.classList.add('active');
    last = idx;
  }}
}});
document.querySelectorAll('.w').forEach(el => {{
  el.addEventListener('click', () => {{
    audio.currentTime = parseFloat(el.dataset.s);
    audio.play();
  }});
}});
</script>
</body>
</html>"""

html_path = Path(CHECK_MP3).with_suffix(".html")
html_path.write_text(html, encoding="utf-8")
print(f"HTML player saved: {html_path.name}")
print(f"\nBoth files are in: {Path(CHECK_MP3).parent}")
print("\n--- Transcript preview (first 20 words) ---")
print("\n".join(lines[:20]))

In [ ]:
# ── Cell 13: Compare source text vs WhisperX transcript ──────────
# Shows word-by-word diff of what was sent to TTS vs what was spoken.
# Mismatches = words the TTS model misread — fix them in the source .txt
# then re-run Cell 9 on that chapter.
#
# Set to the same chapter you transcribed in Cell 12.
# CHAPTER_TXT is the source .txt; TRANSCRIPT_TXT is from Cell 12.

CHAPTER_TXT    = f"{TXT_DIR}/03_03_SEARCH_FOR_MR_HYDE.txt"      # ← source text
TRANSCRIPT_TXT = f"{OUTPUT_DIR}/03_SEARCH_FOR_MR_HYDE.transcript.txt"  # ← from Cell 12

# ─────────────────────────────────────────────────────────────────
import re, difflib, json
from pathlib import Path

# ── Load and tokenise source text ────────────────────────────────
src_raw   = Path(CHAPTER_TXT).read_text(encoding="utf-8")
src_words = re.findall(r"[A-Za-z''\-]+|[^\s]", src_raw)

# ── Load transcript (strip timestamps and flags) ──────────────────
trans_lines = Path(TRANSCRIPT_TXT).read_text(encoding="utf-8").splitlines()
trans_words = []
for line in trans_lines:
    m = re.match(r"\s*[\d.]+s\s+(.+?)(\s+⚠)?$", line)
    if m:
        trans_words.append(m.group(1).strip())

print(f"Source words     : {len(src_words)}")
print(f"Transcript words : {len(trans_words)}")

# ── Word-level diff ───────────────────────────────────────────────
matcher = difflib.SequenceMatcher(None,
    [w.lower().strip(".,!?\"';:-") for w in src_words],
    [w.lower().strip(".,!?\"';:-") for w in trans_words],
    autojunk=False,
)

mismatches = []
for tag, i1, i2, j1, j2 in matcher.get_opcodes():
    if tag == "replace":
        src_chunk   = " ".join(src_words[i1:i2])
        trans_chunk = " ".join(trans_words[j1:j2])
        mismatches.append((src_chunk, trans_chunk))

print(f"\nMismatches found : {len(mismatches)}\n")
print(f"{'SOURCE (intended)':35s}  →  SPOKEN (transcribed)")
print("─" * 70)
for src, spoken in mismatches[:40]:
    print(f"  {src:33s}  →  {spoken}")
if len(mismatches) > 40:
    print(f"  ... and {len(mismatches)-40} more")

# ── Save HTML diff report ─────────────────────────────────────────
rows = "".join(
    f"<tr><td class='src'>{s}</td><td class='arrow'>→</td><td class='spoken'>{t}</td></tr>"
    for s, t in mismatches
)
html = f"""<!DOCTYPE html><html lang="en"><head><meta charset="UTF-8">
<title>Diff: {Path(CHAPTER_TXT).stem}</title>
<style>
  body {{ font-family: monospace; background:#1a1a2e; color:#e0e0e0;
          padding:30px; max-width:900px; margin:auto; }}
  h2   {{ color:#a0c4ff; }}
  table {{ border-collapse:collapse; width:100%; }}
  td   {{ padding:6px 12px; border-bottom:1px solid #333; }}
  .src    {{ color:#ff9999; width:40%; }}
  .arrow  {{ color:#888; width:5%; text-align:center; }}
  .spoken {{ color:#99ff99; width:55%; }}
  th   {{ color:#888; padding:8px 12px; border-bottom:2px solid #555; text-align:left; }}
</style></head><body>
<h2>Source vs Spoken — {Path(CHAPTER_TXT).stem}</h2>
<p style="color:#888">{len(mismatches)} mismatches &nbsp;|&nbsp;
  <span style="color:#ff9999">red = intended</span> &nbsp;|&nbsp;
  <span style="color:#99ff99">green = what was spoken</span></p>
<table><tr><th>Source (intended)</th><th></th><th>Spoken (transcribed)</th></tr>
{rows}
</table></body></html>"""

diff_path = Path(OUTPUT_DIR) / f"{Path(CHAPTER_TXT).stem}_diff.html"
diff_path.write_text(html, encoding="utf-8")
print(f"\nDiff report saved: {diff_path.name}")
print("Download and open in your browser to review all mismatches.")

In [ ]:
# ── Cell 11: Install WhisperX (run once per session) ─────────────
%pip install -q whisperx